In [ ]:
# Libraries Used
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Get Price, Volume, and Fear & Greed Data
price_vol_data = yf.download('ETH-USD', start='2018-02-01', end='2026-09-09')

url = "https://api.alternative.me/fng/"
params = {"limit": "0"}
response = requests.get(url, params=params).json()

# Price and Volume Data
df_price_vol = pd.DataFrame(price_vol_data)
df_price_vol.to_csv("Data/price_vol.csv")

df_price_data = pd.read_csv("Data/price_vol.csv")
df_price_data.drop([0, 1], inplace=True)
df_price_data.reset_index(inplace=True)
df_price_data.drop(columns=["High", "Low", "Open", "index"], inplace=True)
df_price_data.rename(columns={"Price": "Date", "Close": "Price"}, inplace=True)

df_price_data.Date = pd.to_datetime(df_price_data.Date)
df_price_data.Price = pd.to_numeric(df_price_data.Price)
df_price_data.Volume = pd.to_numeric(df_price_data.Volume)

print(f"Date dtype: {df_price_data.Date.dtypes}, "
      f"\nPrice dtype: {df_price_data.Price.dtypes}, "
      f"\nVolume dtype: {df_price_data.Volume.dtypes}")

# Fear and Greed Data
df_fear_greed = pd.DataFrame(response["data"])
df_fear_greed.to_csv("Data/fear_greed.csv")

df_fear_greed.timestamp = pd.to_datetime(df_fear_greed.timestamp, unit="s")
df_fear_greed.value = pd.to_numeric(df_fear_greed.value)

df_fear_greed.sort_values(by="timestamp", inplace=True)
df_fear_greed.reset_index(inplace=True)

df_fear_greed.drop(columns=["time_until_update", "value_classification", "index"], inplace=True)
df_fear_greed.rename(columns={"timestamp": "Date", "value": "FG_Value"}, inplace=True)

print(f"FG_Value dtype: {df_fear_greed.FG_Value.dtype}"
      f"\nDate dtype: {df_fear_greed.Date.dtype}")

# Merge Data
df = pd.merge(df_price_data, df_fear_greed, how="left", on="Date")

print(f"Null values before dropping:\n{df.isnull().sum()}")
df.dropna(inplace=True)

# Scatter Plots
plt.figure(figsize=(8, 4), dpi=200)

with sns.axes_style("darkgrid"):
    price_vol_sp = sns.scatterplot(df,
                                    x="Price",
                                    y="Volume",
                                    hue="Volume",
                                    size="Volume")

price_vol_sp.set(xlim=(0, df.Price.max() * 1.1),
                  ylim=(0, df.Volume.max() * 1.1),
                  xlabel="ETH Price",
                  ylabel="ETH Volume")

plt.savefig("Graphs/price_vol_scatter.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4), dpi=200)

with sns.axes_style("darkgrid"):
    price_fg_sp = sns.scatterplot(df,
                                   x="Price",
                                   y="FG_Value",
                                   hue="FG_Value",
                                   size="FG_Value")

price_fg_sp.set(xlim=(0, df.Price.max() * 1.1),
                 ylim=(0, df.FG_Value.max() * 1.1),
                 xlabel="ETH Price",
                 ylabel="Fear & Greed Value")

plt.savefig("Graphs/price_fg_scatter.png", bbox_inches="tight")
plt.show()

# Regression Plots
plt.figure(figsize=(8, 4), dpi=200)

with sns.axes_style("darkgrid"):
    price_vol_rp = sns.regplot(df,
                                x="Price",
                                y="Volume",
                                scatter_kws={"alpha": 0.4},
                                line_kws={"color": "orange"},
                                color="darkblue")

price_vol_rp.set(xlim=(0, df.Price.max() * 1.1),
                  ylim=(0, df.Volume.max() * 1.1),
                  xlabel="ETH Price",
                  ylabel="ETH Volume")

plt.savefig("Graphs/price_vol_regplot.png", bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 4), dpi=200)

with sns.axes_style("darkgrid"):
    price_fg_rp = sns.regplot(df,
                               x="Price",
                               y="FG_Value",
                               scatter_kws={"alpha": 0.4},
                               line_kws={"color": "orange"},
                               color="darkblue")

price_fg_rp.set(xlim=(0, df.Price.max() * 1.1),
                 ylim=(0, df.FG_Value.max() * 1.1),
                 xlabel="ETH Price",
                 ylabel="Fear & Greed Value")

plt.savefig("Graphs/price_fg_regplot.png", bbox_inches="tight")
plt.show()

# Regression Functions
def single_regression(X, y):
    X = df[X] if isinstance(X, list) else df[[X]]
    y = pd.DataFrame(df, columns=[y])
    regression = LinearRegression()
    regression.fit(X, y)

    return regression.intercept_, regression.coef_, regression.score(X, y)


def train_test(X, y):
    X = df[X] if isinstance(X, list) else df[[X]]
    y = pd.DataFrame(df, columns=[y])

    X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                          train_size=number_to_train,
                                                          shuffle=False)

    regression = LinearRegression()
    regression.fit(X_train, y_train)

    return regression.score(X_test, y_test)

# Data To Split
split_date = pd.to_datetime("2024-01-01")
number_to_train = len(df[df.Date < split_date])

# Combined Variables: Full-Data Fit and Train/Test Split
print("Combined (Volume + FG_Value):")
print(f"Intercept, Coefficients, R Squared: {single_regression(['Volume', 'FG_Value'], 'Price')}")
print(f"R Squared on Test Data: {train_test(['Volume', 'FG_Value'], 'Price')}")

# Single Variables: Full-Data Fit and Train/Test Split
print("Volume alone:")
print(f"Intercept, Coefficient, R Squared: {single_regression('Volume', 'Price')}")
print(f"R Squared on Test Data: {train_test('Volume', 'Price')}")

print("\nFG_Value alone:")
print(f"Intercept, Coefficient, R Squared: {single_regression('FG_Value', 'Price')}")
print(f"R Squared on Test Data: {train_test('FG_Value', 'Price')}")